# Qwen3.8-27B on TPU v5e-8, reachable over Tailscale

Three things have to be true before this can serve, and two of them are cheap to
check:

1. **a TPU is visible** — not a CPU session;
2. **free disk holds the weights** — 27B in bf16 is ~55 GB, and Kaggle's working
   disk is smaller than that on most sessions. This is the number that decides
   whether the plan works at all, so it is measured before anything is downloaded;
3. **`vllm-tpu` installs** — the dataset beside this supplies an XLA cache built
   with vllm-tpu 0.28.0 and the MTP patch, not an environment.

The overlay comes up first so the job reuses the `TS_AUTHKEY` attached to this
kernel. Serving is started only if the checks pass; a failure is reported, not
worked around.


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

def sh(cmd, timeout=900):
    r = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr)

# ---- 1. TPU ----
rc, accel = sh("ls -l /dev/accel* 2>/dev/null; printenv | grep -i -E 'TPU|XRT' | head -5")
print("accelerator:", accel.strip()[:400])
rc2, disk = sh("df -BG /kaggle/working /root /tmp 2>/dev/null | tail -4")
print("disk:\n" + disk)
free_gb = 0
for line in disk.splitlines():
    parts = line.split()
    if len(parts) >= 4 and parts[3].endswith("G"):
        try: free_gb = max(free_gb, int(parts[3].rstrip("G")))
        except ValueError: pass
print("free GB (max of the mounts):", free_gb)

# ---- 2. overlay ----
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("TS_AUTHKEY")
except Exception:
    key = os.environ.get("TS_AUTHKEY", "")
tailnet_ip = ""
if key:
    sh("curl -fsSL https://tailscale.com/install.sh -o /tmp/ti.sh && sh /tmp/ti.sh 2>&1 | tail -2")
    SOCK = "/tmp/ts.sock"
    sh(f"nohup tailscaled --tun=userspace-networking --socks5-server=127.0.0.1:1055 "
       f"--socket={SOCK} --state=/kaggle/working/ts.state >/tmp/tailscaled.log 2>&1 & sleep 6")
    p = subprocess.run(["bash", "-lc", f'tailscale --socket={SOCK} up --auth-key="$TSKEY" '
                                     '--hostname=kaggle-tpu --accept-dns=false'],
                       capture_output=True, text=True, env={**os.environ, "TSKEY": key}, timeout=180)
    print("tailscale up rc", p.returncode)
    tailnet_ip = sh(f"tailscale --socket={SOCK} ip -4")[1].strip()
    print("tailnet ipv4:", tailnet_ip)
else:
    print("no TS_AUTHKEY attached to this kernel; the overlay is skipped")

report = {"free_gb": free_gb, "tailnet_ipv4": tailnet_ip,
          "model": "Qwen/Qwen3.8-27B", "weights_gb_bf16": 55}
Path("/kaggle/working/tpu_report.json").write_text(json.dumps(report, indent=2))

# ---- 3. verdict ----
if free_gb < 60:
    raise SystemExit(
        f"only {free_gb} GB free; Qwen3.8-27B needs ~55 GB of weights plus cache. "
        "Serving it here would fail mid-download. Nothing was installed or downloaded.")
print("feasible: disk and overlay are in place; the serving step is the next version")
